# Experiments 014 and 015: the 7B supervised control (Colab)

The last piece of the 7b column. 014 trains mc-sft from the raw base (8
passes over the 245 prompts, 3 seeds), 015 the same but warm-started from the
exact 013 seed-0 gsm8k adapter. Generative evals for all six checkpoints.

Expectation on record before the run: both match grpo overall (~.80, sub-1pt
seed spread) and do NOT reproduce grpo's hard tier (.639-.657, which sits
7-9pt above the recall ceiling); supervised hard expected ~.57-.59. That
prediction is falsifiable, if supervised also reaches .64+ on hard the
reasoning-dividend claim fails and the memo will say so.

Base generative reference comes from 013 (summaries here keep base_acc null
by design), base probes recompute in minutes. ~6-7h, resume-safe.

If a run OOMs: the attempt moves to a failed_attempts folder and the next
pre-registered config restarts from step zero, full procedure in 7b
plan doc.

## 0. Compute units and mount Drive

Fill `cu_before` and `unit_rate` now.

In [1]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 103.95   # from the resources panel
unit_rate = 6.77   # units per hour, the plan requires it per session
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '014_015_colab_7b_mc_sft', 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260723_194231 | cu_before = 103.95
Mounted at /content/drive
closed previous session: 20260723_084117
saved open session: 20260723_194231
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: A100 with 80GB required

In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime for the 7b supervised runs'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-0374a9a4-8c40-1345-fd2f-09ade82c9b54)
total vram: 79.3 GiB


## 2. Clone or update the repo (automatic)

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 1986, done.
remote: Counting objects: 100% (465/465), done.
remote: Compressing objects: 100% (289/289), done.
remote: Total 1986 (delta 278), reused 358 (delta 176), pack-reused 1521 (from 1)
Receiving objects: 100% (1986/1986), 64.14 MiB | 13.66 MiB/s, done.
Resolving deltas: 100% (1185/1185), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 163.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.5 MB/s eta 0:00:00
deps installed


## 4. W&B key (automatic)

Reminder: colab secrets are granted per notebook, switch on access for
`WANDB_API_KEY` under the key icon before running. This key is only used for experiment tracking.


In [5]:
import getpass
import os

wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None

if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')

os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check the Drive record and free space

In [6]:
import os, shutil
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 60, 'need >=60GB free on drive for the 7b phases'

open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260723_194231.json
drive free: 179 GB


## 6. Build the data splits

In [7]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 22.6MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.31M [00:00<?, ?B/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:01<00:00, 1.60MB/s,  223kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:01<00:00, 1.60MB/s,  224kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:01<00:00, 319kB/s, 40.8kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:01<00:00, 319kB/s, 40.8kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 379646.97 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 324874.45 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 81.8MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 118MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00:00<?, ?B

## 7. Preflight: the 013 stage-1 adapter must be on Drive

015 chains from the 013 seed-0 gsm8k-sft checkpoint.

In [8]:
import os
D = '/content/drive/MyDrive/inm363'
rel = 'checkpoints/013_COLAB_FULL_B7_sft_grpo_seed0/sft'
for fname in ('adapter_model.safetensors', 'adapter_config.json', 'adapter_base.json'):
    p = os.path.join(D, rel, fname)
    assert os.path.exists(p), f'stage-1 file missing: {p}'
print('warm start present and complete')

warm start present and complete


## 8. Wiring check

In [9]:
!python scripts/run.py --config configs/014_p8_colab_b7.yaml

env=COLAB mode=FULL scale=B7 condition=SFT_ONLY
model=Qwen/Qwen2.5-7B device=cuda stages=['base_eval', 'sft', 'sft_eval']
steps=123 N=8 seed=0
trackers=['tensorboard', 'wandb']
results -> /content/drive/MyDrive/inm363/results/014_p8_COLAB_FULL_B7_sft_only_seed0/  checkpoints -> /content/drive/MyDrive/inm363/checkpoints/014_p8_COLAB_FULL_B7_sft_only_seed0/
local smoke wiring ok


## 9. The training sweeps, probe-only

6 small runs, minutes each. 015's first run does the one-off merge, seeds 1
and 2 reuse the shared merged model.

**If colab disconnects:** re-run the cell, finished runs skip in seconds.

In [10]:
# both cost fields must be filled before anything burns gpu time
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
# python loop with ! calls so the logs stream live, exit code checked so a
# failed run stops the sweep instead of hiding behind the next one
for c in ('014_p8_colab_b7', '015_wsft_colab_b7'):
    for s in (0, 1, 2):
        print(f'===== {c} seed {s} =====')
        !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/{c}.yaml --seed {s} --stage all
        if _exit_code != 0:
            raise SystemExit(f'{c} seed {s} failed with exit code {_exit_code}')

===== 014_p8_colab_b7 seed 0 =====
seed overridden to 0
2026-07-23 19:48:42,133 INFO using wandb key from env
2026-07-23 19:48:42,135 INFO run 014_p8_COLAB_FULL_B7_sft_only_seed0 device=cuda stages=['base_eval', 'sft', 'sft_eval'] trackers=['tensorboard', 'wandb'] out=/content/drive/MyDrive/inm363/results/014_p8_COLAB_FULL_B7_sft_only_seed0
2026-07-23 19:48:42,135 INFO base_eval skipped, probe-only run
config.json: 100% 686/686 [00:00<00:00, 3.97MB/s]
tokenizer_config.json: 100% 7.23k/7.23k [00:00<00:00, 23.6MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 109MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 114MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 129MB/s]
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 79.3MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.95G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.0

## 10. Generative evals, all six checkpoints

In [11]:
# this cell can run on its own, so it carries its own cost-field guard
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
D = '/content/drive/MyDrive/inm363'
jobs = []
for prefix, cfg in (('014_p8', 'configs/014_p8_colab_b7.yaml'), ('015_wsft', 'configs/015_wsft_colab_b7.yaml')):
    for s in (0, 1, 2):
        jobs.append((f'{prefix}_COLAB_FULL_B7_sft_only_seed{s}', cfg, s))
for tag, cfg, s in jobs:
    print(f'===== gen eval {tag} =====')
    !python scripts/eval_adapter.py --config {cfg} --adapter {D}/checkpoints/{tag}/sft --condition sft --out {D}/results/{tag} --seed {s}
    if _exit_code != 0:
        raise SystemExit(f'{tag} eval failed with exit code {_exit_code}')

===== gen eval 014_p8_COLAB_FULL_B7_sft_only_seed0 =====
Loading weights: 100% 339/339 [00:00<00:00, 5570.23it/s]
sft eval (1884 items): 100% 118/118 [05:11<00:00,  2.64s/it, acc 0.786 fmt 1.000]
sft probe already done, skipping
summary completed with the new evals
gen per-tier: {'easy': 0.8953356086461889, 'medium': 0.7799671592775042, 'hard': 0.553030303030303}
probe per-tier: {'easy': 0.8964732650739476, 'medium': 0.812807881773399, 'hard': 0.5656565656565656}
===== gen eval 014_p8_COLAB_FULL_B7_sft_only_seed1 =====
Loading weights: 100% 339/339 [00:00<00:00, 5148.53it/s]
sft eval (1884 items): 100% 118/118 [05:12<00:00,  2.65s/it, acc 0.780 fmt 1.000]
sft probe already done, skipping
summary completed with the new evals
gen per-tier: {'easy': 0.888509670079636, 'medium': 0.7783251231527094, 'hard': 0.5429292929292929}
probe per-tier: {'easy': 0.8976109215017065, 'medium': 0.812807881773399, 'hard': 0.5757575757575758}
===== gen eval 014_p8_COLAB_FULL_B7_sft_only_seed2 =====
Loading

## 11. Peek at the results

GRPO references for the hard tier: .639 / .657 (sft_grpo seeds), .654
(base_grpo). The question is whether the numbers below get anywhere near.

In [12]:
import json, os
D = '/content/drive/MyDrive/inm363'
for prefix in ('014_p8', '015_wsft'):
    for s in (0, 1, 2):
        tag = f'{prefix}_COLAB_FULL_B7_sft_only_seed{s}'
        d = os.path.join(D, 'results', tag)
        lp_f = os.path.join(d, 'sft_logprob.json')
        ev_f = os.path.join(d, 'sft_eval.json')
        lp = json.load(open(lp_f))['per_tier_acc'] if os.path.exists(lp_f) else None
        ev = json.load(open(ev_f))['per_tier_acc'] if os.path.exists(ev_f) else None
        fmt = lambda x: {k: round(v, 3) for k, v in x.items()} if x else '-'
        print(f'{tag}  probe {fmt(lp)}  gen {fmt(ev)}')

014_p8_COLAB_FULL_B7_sft_only_seed0  probe {'easy': 0.896, 'medium': 0.813, 'hard': 0.566}  gen {'easy': 0.895, 'medium': 0.78, 'hard': 0.553}
014_p8_COLAB_FULL_B7_sft_only_seed1  probe {'easy': 0.898, 'medium': 0.813, 'hard': 0.576}  gen {'easy': 0.889, 'medium': 0.778, 'hard': 0.543}
014_p8_COLAB_FULL_B7_sft_only_seed2  probe {'easy': 0.898, 'medium': 0.81, 'hard': 0.568}  gen {'easy': 0.891, 'medium': 0.778, 'hard': 0.558}
015_wsft_COLAB_FULL_B7_sft_only_seed0  probe {'easy': 0.901, 'medium': 0.814, 'hard': 0.566}  gen {'easy': 0.874, 'medium': 0.796, 'hard': 0.518}
015_wsft_COLAB_FULL_B7_sft_only_seed1  probe {'easy': 0.9, 'medium': 0.81, 'hard': 0.566}  gen {'easy': 0.881, 'medium': 0.787, 'hard': 0.53}
015_wsft_COLAB_FULL_B7_sft_only_seed2  probe {'easy': 0.901, 'medium': 0.806, 'hard': 0.563}  gen {'easy': 0.877, 'medium': 0.79, 'hard': 0.548}


## 12. Close the units record

In [14]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 90.77   # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

logged: {'session_id': '20260723_194231', 'ts_start': '2026-07-23 19:42', 'notebook': '014_015_colab_7b_mc_sft', 'colab_cu_before': 103.95, 'unit_rate': 6.77, 'ts_end': '2026-07-23 21:42', 'colab_cu_after': 90.77, 'colab_cu_used': 13.18, 'closed_on_resume': False}
